# Классификация кристаллических структур урановых соединений

## Расширенный датасет:
- **NaCl** (7x7x7) - структура хлорида натрия
- **UN2** - динитрид урана
- **U2N3 (alpha)** - кубическая фаза (Ia-3)
- **U2N3 (beta)** - тригональная фаза (P-3m1)
- **UC** - монокарбид урана (NaCl-type)
- **UC2 (alpha)** - тетрагональная фаза (I4/mmm)
- **UC2 (beta)** - кубическая фаза (Fm-3m)
- **U2C3** - сесквикарбид урана (I-43d)


In [10]:
from google.colab import files
import zipfile
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка архива
#uploaded = files.upload()
#archive_name = list(uploaded.keys())[0]

# Распаковка архива
#with zipfile.ZipFile(archive_name, 'r') as zip_ref:
#    zip_ref.extractall('data')

data_dir = 'data'

def load_lattice_data(lattice_dir, label):
    """
    Загрузка данных для одного типа решётки
    """
    lattice_data = []
    lattice_labels = []
    for lattice_num in os.listdir(lattice_dir):
        lattice_path = os.path.join(lattice_dir, lattice_num)
        if os.path.isdir(lattice_path):
            kde_data = []
            for filename in os.listdir(lattice_path):
                filepath = os.path.join(lattice_path, filename)
                kde_values = pd.read_csv(filepath, header=0).iloc[:, 0].values[1:]
                kde_data.append(kde_values.astype(float))
            kde_matrix = np.column_stack(kde_data)
            lattice_data.append(kde_matrix)
            lattice_labels.append(label)
    return lattice_data, lattice_labels

# Загрузка данных для каждого типа решётки
print("Загрузка данных...")

# Оригинальные структуры
lattice_data_NaCl, labels_NaCl = load_lattice_data(
    os.path.join(data_dir, '7x7x7'), 'NaCl')
lattice_data_UN2, labels_UN2 = load_lattice_data(
    os.path.join(data_dir, 'UN2'), 'UN2')
lattice_data_U2N3, labels_U2N3 = load_lattice_data(
    os.path.join(data_dir, 'U2N3'), 'U2N3')

# Новые структуры нитридов
lattice_data_U2N3_alpha, labels_U2N3_alpha = load_lattice_data(
    os.path.join(data_dir, 'U2N3_alpha'), 'U2N3_alpha')
lattice_data_U2N3_beta, labels_U2N3_beta = load_lattice_data(
    os.path.join(data_dir, 'U2N3_beta'), 'U2N3_beta')

# Карбиды урана
lattice_data_UC, labels_UC = load_lattice_data(
    os.path.join(data_dir, 'UC'), 'UC')
lattice_data_UC2_alpha, labels_UC2_alpha = load_lattice_data(
    os.path.join(data_dir, 'UC2_alpha'), 'UC2_alpha')
lattice_data_UC2_beta, labels_UC2_beta = load_lattice_data(
    os.path.join(data_dir, 'UC2_beta'), 'UC2_beta')
lattice_data_U2C3, labels_U2C3 = load_lattice_data(
    os.path.join(data_dir, 'U2C3'), 'U2C3')

# Объединение всех данных
X = (lattice_data_NaCl + lattice_data_UN2 + lattice_data_U2N3 +
     lattice_data_U2N3_alpha + lattice_data_U2N3_beta +
     lattice_data_UC + lattice_data_UC2_alpha + lattice_data_UC2_beta +
     lattice_data_U2C3)

y = np.hstack([
    np.array(labels_NaCl),
    np.array(labels_UN2),
    np.array(labels_U2N3),
    np.array(labels_U2N3_alpha),
    np.array(labels_U2N3_beta),
    np.array(labels_UC),
    np.array(labels_UC2_alpha),
    np.array(labels_UC2_beta),
    np.array(labels_U2C3)
])

print(f"\nВсего загружено образцов: {len(X)}")
print(f"Распределение по классам:")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count}")

def pad_matrix(matrix, max_columns):
    """
    Дополнение матрицы нулями до нужного размера
    """
    padded_matrix = np.pad(
        matrix, ((0, 0), (0, max_columns - matrix.shape[1])),
        mode='constant'
    )
    return padded_matrix

# Находим максимальное количество ионов в одной решётке
max_ions = max(matrix.shape[1] for matrix in X)
print(f"\nМаксимальное число ионов в решётке: {max_ions}")

# Дополняем все матрицы до одинакового размера
X_padded = [pad_matrix(matrix, max_ions) for matrix in X]

# Уплощаем матрицы в вектора
def matrix_to_vector(matrix):
    return matrix.flatten()

X_flattened = [matrix_to_vector(matrix) for matrix in X_padded]
X_flattened = np.array(X_flattened)

print(f"Размер признакового пространства: {X_flattened.shape}")

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_flattened, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")


Загрузка данных...

Всего загружено образцов: 180
Распределение по классам:
  NaCl: 20
  U2C3: 20
  U2N3: 20
  U2N3_alpha: 20
  U2N3_beta: 20
  UC: 20
  UC2_alpha: 20
  UC2_beta: 20
  UN2: 20

Максимальное число ионов в решётке: 343
Размер признакового пространства: (180, 342657)

Размер обучающей выборки: 144
Размер тестовой выборки: 36


In [11]:
# Создание и обучение модели
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Предсказание на тестовой выборке
y_pred = model.predict(X_test)

# Оценка точности
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Подбор гиперпараметров
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")

# Обучение модели с лучшими параметрами
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print(f"Best Accuracy: {accuracy_best:.2f}")

# Использование других моделей
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM Accuracy: {accuracy_svm:.2f}")

knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"KNN Accuracy: {accuracy_knn:.2f}")

# Сохранение модели
joblib.dump(best_model, 'kde_classifier.pkl')

Accuracy: 1.00
Лучшие параметры: {'max_depth': None, 'n_estimators': 50}
Best Accuracy: 1.00
SVM Accuracy: 0.97
KNN Accuracy: 0.86


['kde_classifier.pkl']

In [13]:
import os
import pandas as pd
import numpy as np
import joblib

# Путь к уже распакованной папке с данными
new_data_dir = 'data'  # или другой путь, где лежат ваши данные

def load_lattice_data(lattice_dir):
    kde_data = []
    for ion_file in os.listdir(lattice_dir):
        filepath = os.path.join(lattice_dir, ion_file)
        if os.path.isfile(filepath) and ion_file.endswith('.csv'):
            try:
                # Чтение CSV файла с заголовком и пропуск первой строки
                kde_values = pd.read_csv(filepath, header=0).iloc[:, 0].values[1:].astype(float)
                kde_data.append(kde_values)
                print(f"Успешно загружен файл: {ion_file}, размер: {len(kde_values)}")
            except Exception as e:
                print(f"Ошибка при чтении файла {filepath}: {e}")

    if not kde_data:
        raise ValueError(f"Не найдено ни одного корректного файла в папке {lattice_dir}")

    kde_matrix = np.column_stack(kde_data)
    print(f"Создана матрица размером: {kde_matrix.shape}")
    return kde_matrix

# Путь к конкретной папке с решеткой
lattice_dir = os.path.join(new_data_dir, '21')

# Проверка существования папки
if not os.path.exists(lattice_dir):
    print(f"Папка {lattice_dir} не существует!")
    print("Доступные папки в new_data:")
    for item in os.listdir(new_data_dir):
        item_path = os.path.join(new_data_dir, item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}")
            # Показать подпапки
            for subitem in os.listdir(item_path):
                subitem_path = os.path.join(item_path, subitem)
                if os.path.isdir(subitem_path):
                    print(f"    📂 {subitem}")
else:
    print(f"Путь к папке с данными: {lattice_dir}")
    print(f"Список файлов в папке: {os.listdir(lattice_dir)[:10]}...")  # Показываем первые 10 файлов

    # Загрузка данных для новой решётки
    try:
        kde_matrix = load_lattice_data(lattice_dir)
    except Exception as e:
        print(f"Ошибка при загрузке данных: {e}")
        raise

    # Загрузка сохранённой модели
    model = joblib.load('kde_classifier.pkl')

    def pad_matrix(matrix, max_columns):
        padded_matrix = np.pad(matrix, ((0, 0), (0, max_columns - matrix.shape[1])), mode='constant')
        return padded_matrix

    # Максимальное количество ионов в одной решётке из обучающей выборки
    max_ions = 343  # Замените на актуальное значение из вашей обученной модели

    # Дополняем матрицу до одинакового размера
    kde_matrix_padded = pad_matrix(kde_matrix, max_ions)
    print(f"Матрица после дополнения: {kde_matrix_padded.shape}")

    # Уплощаем матрицу
    def matrix_to_vector(matrix):
        return matrix.flatten()

    kde_vector = matrix_to_vector(kde_matrix_padded)
    kde_vector = kde_vector.reshape(1, -1)  # Преобразуем в двумерный массив (одна строка)
    print(f"Вектор для классификации: {kde_vector.shape}")

    # Предсказание типа решётки
    predicted_label = model.predict(kde_vector)
    prediction_proba = model.predict_proba(kde_vector)

    print(f"Предсказанный тип решетки: {predicted_label[0]}")
    print(f"Вероятности:")

    for i, (class_name, prob) in enumerate(zip(model.classes_, prediction_proba[0])):
        print(f"  {class_name}: {prob:.4f} ({prob*100:.2f}%)")

Папка data/21 не существует!
Доступные папки в new_data:
  📁 UC
    📂 6
    📂 19
    📂 3
    📂 17
    📂 5
    📂 4
    📂 13
    📂 2
    📂 1
    📂 15
    📂 11
    📂 12
    📂 7
    📂 8
    📂 9
    📂 14
    📂 20
    📂 10
    📂 18
    📂 16
  📁 7x7x7_v2
  📁 UC2_beta
    📂 6
    📂 19
    📂 3
    📂 17
    📂 5
    📂 4
    📂 13
    📂 2
    📂 1
    📂 15
    📂 11
    📂 12
    📂 7
    📂 8
    📂 9
    📂 14
    📂 20
    📂 10
    📂 18
    📂 16
  📁 U2C3
    📂 6
    📂 19
    📂 3
    📂 17
    📂 5
    📂 4
    📂 13
    📂 2
    📂 1
    📂 15
    📂 11
    📂 12
    📂 7
    📂 8
    📂 9
    📂 14
    📂 20
    📂 10
    📂 18
    📂 16
  📁 U2N3_alpha
    📂 6
    📂 19
    📂 3
    📂 17
    📂 5
    📂 4
    📂 13
    📂 2
    📂 1
    📂 15
    📂 11
    📂 12
    📂 7
    📂 8
    📂 9
    📂 14
    📂 20
    📂 10
    📂 18
    📂 16
  📁 UC2_alpha
    📂 6
    📂 19
    📂 3
    📂 17
    📂 5
    📂 4
    📂 13
    📂 2
    📂 1
    📂 15
    📂 11
    📂 12
    📂 7
    📂 8
    📂 9
    📂 14
    📂 20
    📂 10
    📂 18
    📂 16
  📁 UN2
    📂 6
 